# EvoCA — exploration notebook

Tests the C core (ctypes), verifies exact GoL, and launches the SDL2 interactive display.

Run cells in order.  **Build first**, then proceed.

## 1  Build

In [1]:
import subprocess, sys, os
root = os.path.abspath('')
# macOS: gcc -dynamiclib; Linux: gcc -shared
import platform
flag = '-dynamiclib' if platform.system() == 'Darwin' else '-shared'
ext  = 'dylib'      if platform.system() == 'Darwin' else 'so'
cmd  = ['gcc', '-O2', '-Wall', '-fPIC', flag,
        '-o', f'C/libevoca.{ext}', 'C/evoca.c']
r = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(r.stdout or '(no stdout)')
if r.returncode != 0:
    print('STDERR:', r.stderr, file=sys.stderr)
    raise RuntimeError('Build failed')
print('Build OK')

(no stdout)
Build OK


## 2  Imports

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import time
from pprint import pprint

from python.evoca_py import EvoCA, make_gol_lut, LUT_BYTES, lut_bit_index
from python.evoca_py import unpack_lut, available_state_init
from python.display  import run as sdl_run
from python.controls import run_with_controls
from python.evoca_py import import_run
from python.controls import available_probes
from python.evoca_explore import evoca_from_scan, evoca_from_scan_top

# Scan imports

End-to-end works. From notebook you can now do:
```
from evoca_explore import evoca_from_scan, evoca_from_scan_top
from evoca_py import import_run
from controls import run_with_controls
```
### Option A: pick a specific config_idx from the CSV
```
path = evoca_from_scan('Scans/2026-04-27_initial', config_idx=49,
                     descriptor='best_corr_length',
                     probes={'activity': True, 'q_activity': True,
                             'n_activity': True, 'env_food': True,
                             'priv_food': True, 'ts': True},
                     colormode=4)
sim, kw = import_run(path)
run_with_controls(sim, **kw)
```
### Option B: dump top-K by composite score
```
results = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=5,
                            probes={...}, colormode=4)
```
**results is [(config_idx, path), ...]**
```
sim, kw = import_run(results[0][1])
run_with_controls(sim, **kw)
```

In [4]:
scan = pd.read_csv('Scans/2026-04-27_initial/results.csv')

In [5]:
scan.head()

,F_std_final,F_std_mean,F_std_temporal_std,N,alive_density_final,alive_density_mean,alive_density_temporal_std,config_idx,correlation_length_final,correlation_length_mean,...,param_food_inc,param_gdiff,param_m_scale,param_mu_egenome,param_mu_lut,param_restricted_mu,param_tax,seed,shadow,unique_top_genomes
0,0.000000,0.015025,0.039278,256,0.000000,0.011225,0.069448,11,0.0,9.529412,...,0.015,0.010,0.6,0.001,0.030,True,0.035,0,True,16
1,0.000000,0.026440,0.068274,256,0.000000,0.013800,0.070772,4,0.0,8.039216,...,0.015,0.005,0.4,0.001,0.030,False,0.025,0,True,14
2,0.000000,0.029890,0.068056,256,0.000000,0.014282,0.071805,13,0.0,11.705882,...,0.015,0.005,0.6,0.001,0.010,False,0.025,0,True,25
3,0.451295,0.437447,0.065362,256,0.495056,0.467331,0.090361,0,16.0,25.980392,...,0.015,0.020,0.6,0.001,0.003,True,0.025,0,True,47
4,0.017998,0.033617,0.073552,256,0.999969,0.980325,0.084667,8,1.0,8.333333,...,0.025,0.020,0.4,0.001,0.030,False,0.025,0,True,1


In [6]:
scan.columns

Index(['F_std_final', 'F_std_mean', 'F_std_temporal_std', 'N',
       'alive_density_final', 'alive_density_mean',
       'alive_density_temporal_std', 'config_idx', 'correlation_length_final',
       'correlation_length_mean', 'correlation_length_temporal_std', 'error',
       'excess_activity_final', 'excess_activity_slope', 'extinct',
       'final_pop', 'init', 'largest_patch_final', 'largest_patch_mean',
       'largest_patch_temporal_std', 'min_pop', 'n_distinct_genomes_final',
       'n_distinct_genomes_mean', 'n_distinct_genomes_temporal_std',
       'n_patches_final', 'n_patches_mean', 'n_patches_temporal_std',
       'n_steps', 'param_food_inc', 'param_gdiff', 'param_m_scale',
       'param_mu_egenome', 'param_mu_lut', 'param_restricted_mu', 'param_tax',
       'seed', 'shadow', 'unique_top_genomes'],
      dtype='str')

In [3]:
path = evoca_from_scan('Scans/2026-04-27_initial', config_idx=49,
                     descriptor='best_corr_length',
                     probes={'activity': True, 'q_activity': True,
                             'n_activity': True, 'env_food': True,
                             'priv_food': True, 'ts': True},
                     colormode=4)
sim, kw = import_run(path)
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6291386368)>

In [7]:
results = evoca_from_scan_top('Scans/2026-04-27_initial', top_k=10,
                            probes={'activity': True,
                                   'ts':True}, colormode=4)

In [8]:
results

[(49, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_1_cfg49_score0.75.evoca'),
 (30, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_2_cfg30_score0.72.evoca'),
 (70, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_3_cfg70_score0.68.evoca'),
 (34, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_4_cfg34_score0.63.evoca'),
 (35, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_5_cfg35_score0.62.evoca'),
 (3, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_6_cfg3_score0.61.evoca'),
 (87, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_7_cfg87_score0.60.evoca'),
 (111, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_8_cfg111_score0.58.evoca'),
 (82, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_9_cfg82_score0.55.evoca'),
 (39, '/Users/n/Projects/EvoCA/Runs/2026-04-27_top_10_cfg39_score0.55.evoca')]

In [11]:
sim, kw = import_run(results[3][1])
run_with_controls(sim, **kw)

<Thread(evoca-sim, started daemon 6341865472)>